In [1]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

from dotenv import load_dotenv
import os
import pandas as pd
import json

# ---------------------------
# Load environment variables
# ---------------------------
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_KEY")

# ---------------------------
# Load Vectorstore + Retriever
# ---------------------------
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

persist_directory = "zoomcamp_transcripts/chroma_store"
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# ---------------------------
# Initialize OpenAI LLM
# ---------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",  # can switch to gpt-4o, gpt-4-turbo, etc.
    temperature=0
)

# ---------------------------
# Custom Prompt for Concise Answers
# ---------------------------
prompt_template = """
You are a helpful assistant. 
Use the provided context to answer the question concisely in 2–3 sentences. 
If the context does not contain the answer, say "I don’t know."

Context:
{context}

Question:
{question}

Answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

# ---------------------------
# Step 1: Load ragas_dataset.json
# ---------------------------
with open("ragas_dataset.json", "r", encoding="utf-8") as f:
    ragas_data = json.load(f)

questions = [item["question"] for item in ragas_data]
ground_truths = [item["answer"] for item in ragas_data]
contexts_list = [item["contexts"] for item in ragas_data]

# ---------------------------
# Step 2: Run queries through your RAG system
# ---------------------------
answers = []
retrieved_contexts = []

for q in questions:
    result = qa_chain({"query": q})
    answers.append(result["result"])
    retrieved_contexts.append([doc.page_content for doc in result["source_documents"]])

# ---------------------------
# Step 3: Prepare dataset for Ragas
# ---------------------------
dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": retrieved_contexts,
    "ground_truth": ground_truths,
})

# ---------------------------
# Step 4: Run Ragas evaluation
# ---------------------------
result = evaluate(
    dataset=dataset,
    metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
)

print("📊 Evaluation Results:")
print(result)

# ---------------------------
# Step 5: Convert to Pandas + Export to Excel
# ---------------------------
df_metrics = result.to_pandas()

# Save original dataset alongside metrics
df_full = pd.concat([dataset.to_pandas(), df_metrics], axis=1)

# Export to Excel
output_file = "ragas_evaluation.xlsx"
df_full.to_excel(output_file, index=False)

print(f"✅ Results exported to {output_file}")


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_36378/678322984.py:30: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/tmp/ipykernel_36378/678322984.py:33: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used 

📊 Evaluation Results:
{'context_precision': 0.3543, 'context_recall': 0.4571, 'faithfulness': 0.7458, 'answer_relevancy': 0.7377}
✅ Results exported to ragas_evaluation.xlsx


In [3]:
import json
import os
from openai import OpenAI


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_KEY")


client = OpenAI(api_key=OPENAI_API_KEY)



# # Load OpenAI key
# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_KEY")
# client = OpenAI(api_key=OPENAI_API_KEY)

# Load all transcripts (example: multiple videos)
with open("zoomcamp_transcripts/zoomcamp_chunks_30s.json", "r", encoding="utf-8") as f:
    transcripts = json.load(f)

qa_pairs = []

for video in transcripts:
    video_title = video["video_title"]
    context_text = video["text"]
    
    prompt = f"""
You are an expert at creating educational content from video transcripts.
Given the transcript below from a YouTube video titled "{video_title}", generate 2–3 high-quality, diverse question-answer pairs.

Transcript:
{context_text}

Requirements:
- Questions should cover concepts, how-to steps, terminology, or examples.
- Answers must be concise, correct, and factual.
- Return output as a JSON array of objects with "question" and "answer".
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )

    content = response.choices[0].message.content.strip()
    try:
        pairs = json.loads(content)
        qa_pairs.extend(pairs)
    except Exception as e:
        print(f"Failed to parse JSON for video {video_title}: {e}")

# Save all generated Q&A pairs
with open("ragas_dataset_diverse.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=2, ensure_ascii=False)

print(f"✅ Generated {len(qa_pairs)} diverse Q&A pairs")

Failed to parse JSON for video How to Build Agentic RAG Pipelines with OpenAI Function Calling? - LLM Zoomcamp Bonus Module: Expecting value: line 1 column 1 (char 0)
Failed to parse JSON for video How to Build Agentic RAG Pipelines with OpenAI Function Calling? - LLM Zoomcamp Bonus Module: Expecting value: line 1 column 1 (char 0)
Failed to parse JSON for video How to Build Agentic RAG Pipelines with OpenAI Function Calling? - LLM Zoomcamp Bonus Module: Expecting value: line 1 column 1 (char 0)
Failed to parse JSON for video How to Build Agentic RAG Pipelines with OpenAI Function Calling? - LLM Zoomcamp Bonus Module: Expecting value: line 1 column 1 (char 0)
Failed to parse JSON for video How to Build Agentic RAG Pipelines with OpenAI Function Calling? - LLM Zoomcamp Bonus Module: Expecting value: line 1 column 1 (char 0)


KeyboardInterrupt: 

In [ ]:
#try diversity
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
from textwrap import shorten

# ---------------------------
# Load OpenAI API key
# ---------------------------
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_KEY not found in environment variables!")

client = OpenAI(api_key=OPENAI_API_KEY)

# ---------------------------
# Load your transcripts JSON
# ---------------------------
# with open("zoomcamp_transcripts/zoomcamp_chunks_30s.json", "r", encoding="utf-8") as f:
#     transcripts = json.load(f)
with open("zoomcamp_transcripts/zoomcamp_chunks_30s.json", "r", encoding="utf-8") as f:
    videos = json.load(f)

# ---------------------------
# Helper function to parse JSON safely
# ---------------------------
def parse_response(content):
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        # Attempt to extract JSON from text
        start = content.find("[")
        end = content.rfind("]") + 1
        try:
            return json.loads(content[start:end])
        except:
            return []  # fallback if still fails

# ---------------------------
# Generate Q&A pairs
# ---------------------------
qa_pairs = []

for video in videos:
    video_title = video["video_title"]
    context_text = video["text"]

    # Truncate context to 500 words max (optional, adjust as needed)
    context_text = " ".join(context_text.split()[:500])

    prompt = f"""
You are an expert at creating educational content from video transcripts.
Given the transcript below from a YouTube video titled "{video_title}", generate 2–3 high-quality, diverse question-answer pairs.

Transcript:
{context_text}

Requirements:
- Questions should cover concepts, how-to steps, terminology, or examples.
- Answers must be concise, correct, and factual.
- Return output as a JSON array ONLY, with objects containing "question" and "answer".
- DO NOT include any extra text or explanations outside the JSON array.
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=600
        )
        content = response.choices[0].message.content.strip()
        pairs = parse_response(content)
        for pair in pairs:
            # attach video metadata for reference
            pair["video_title"] = video_title
            pair["video_url"] = video.get("video_url")
            qa_pairs.append(pair)

    except Exception as e:
        print(f"⚠️ Failed for video {video_title}: {e}")
        continue

# ---------------------------
# Save Q&A pairs to JSON
# ----------------------0-----
with open("ragas_qa_pairs.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, ensure_ascii=False, indent=2)

print(f"✅ Generated {len(qa_pairs)} Q&A pairs and saved to ragas_qa_pairs.json")

KeyboardInterrupt: 

In [6]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv
import os

# -----------------------------
# Load API Key
# -----------------------------
load_dotenv()
GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("❌ GEMINI_API_KEY not found. Please set it in your .env file.")

# -----------------------------
# Embedding Model & Vectorstore
# -----------------------------
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

persist_directory = "zoomcamp_transcripts/chroma_store"
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# -----------------------------
# LLM Setup (Gemini)
# -----------------------------
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash-latest",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2  # more focused answers
)

# -----------------------------
# Custom Prompt Template
# -----------------------------


prompt_template = """
You're a strict teaching assistant. 
Use ONLY the CONTEXT provided to answer the QUESTION. 
- If the answer is not in the CONTEXT, say: "I don’t know based on the course materials."
- Keep answers concise (2–3 sentences).
- Cite video timestamps when available.

Context:
{context}

Question:
{question}

Answer:
"""
prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["question", "context"]
)

# -----------------------------
# RetrievalQA Chain
# -----------------------------
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

# -----------------------------
# Query Function
# -----------------------------
def ask_question(query: str):
    """Query the RAG system and return structured answer + sources."""
    result = qa_chain({"query": query})

    answer = result["result"]
    sources = [
        {
            "video_title": doc.metadata.get("video_title"),
            "video_url": doc.metadata.get("video_url"),
            "start_time": doc.metadata.get("start_time"),
            "end_time": doc.metadata.get("end_time"),
        }
        for doc in result["source_documents"]
    ]
    return answer, sources

# -----------------------------
# Example Run
# -----------------------------
if __name__ == "__main__":
    query = "How do I set up a virtual environment in Python?"
    answer, sources = ask_question(query)

    print("\n🧠 Answer:\n")
    print(answer)

    print("\n📚 Sources:\n")
    for s in sources:
        print(f"- {s['video_title']} ({s['video_url']} at {s['start_time']}s–{s['end_time']}s)")


/home/codespace/.local/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



🧠 Answer:

The provided text mentions using `python3.9` to create a virtual environment,  and also mentions `vm` and `conda` as packages for creating environments.  More specific instructions on how to set up the environment are not given.

📚 Sources:

- MLOps Zoomcamp 2022 - Office Hours #4 (https://www.youtube.com/watch?v=mQVyA98dlak at 1059.76s–1091.76s)
- ML Zoomcamp 7.7 - (Optional) Advanced Example: Deploying Stable Diffusion Model (https://www.youtube.com/watch?v=NMIi_DDVxAs at 96.54s–128.34s)
- ML Zoomcamp 5.5 - Python Virtual Environment: Pipenv (https://www.youtube.com/watch?v=BMXh8JGROHM at 341.919s–376.479s)
- MLOps Zoomcamp 6.1 - Testing Python code with pytest (https://www.youtube.com/watch?v=CJp1eFQP5nk at 281.28s–314.96s)
- ML Zoomcamp 5.5 - Python Virtual Environment: Pipenv (https://www.youtube.com/watch?v=BMXh8JGROHM at 841.6s–876.8s)
